In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="06-video-recommendation",
    notebook_name="01_recommendation_system_design.ipynb"
)

# Video Recommendation System Design

## The Big Idea (For a 12-Year-Old)

Imagine you walk into the **world's biggest video store** -- it has **10 BILLION videos**. That is more videos than there are people on Earth! You cannot possibly browse through all of them. You need a super-smart assistant who:

- Knows everything you have ever watched, liked, and searched for
- Knows what millions of people similar to you also enjoyed
- Can pick the **perfect 20-50 videos** just for YOU in under 200 milliseconds (faster than you can blink!)

That is what YouTube, Netflix, and TikTok do. **Over 70% of YouTube watch time comes from recommended videos**, not search. This system is worth billions of dollars.

---

## Staff-Level Technical Summary

We design a **YouTube-style homepage video recommendation system** using:
- **Hybrid filtering**: collaborative filtering (candidate generation) + content-based (ranking)
- **Matrix factorization** with WALS optimization for fast candidate generation
- **Two-tower neural network** for flexible, feature-rich candidate generation
- **Multi-stage serving pipeline**: candidate generation -> scoring -> re-ranking
- **ANN (Approximate Nearest Neighbor)** search for sub-linear retrieval
- **Offline**: precision@k, mAP, diversity | **Online**: CTR, watch time, explicit feedback

## 1. Problem Clarification (The Interview Starts Here)

In a real ML system design interview, you ALWAYS start by asking clarifying questions. Never dive into architecture without understanding the requirements first.

### The Interview Dialogue

In [ ]:
# Let's organize the clarifying requirements as structured data
# This is how you'd think about it systematically in an interview

interview_dialogue = [
    {
        "candidate": "Can I assume the business objective is to increase user engagement?",
        "interviewer": "That's correct.",
        "why_ask": "Engagement vs. revenue vs. retention changes the ML objective entirely"
    },
    {
        "candidate": "Does the system recommend similar videos to one being watched, or personalized videos on the homepage?",
        "interviewer": "Homepage video recommendation -- personalized videos when users load the homepage.",
        "why_ask": "'Related videos' has context (current video). Homepage has NO context -- much harder!"
    },
    {
        "candidate": "Since YouTube is global, can I assume users worldwide and videos in different languages?",
        "interviewer": "Yes, fair assumption.",
        "why_ask": "Language and locale features become critical for ranking"
    },
    {
        "candidate": "Can I use interaction data -- clicks, watches, likes, impressions?",
        "interviewer": "Yes, sounds good.",
        "why_ask": "Determines what signals we can use for relevance labels"
    },
    {
        "candidate": "Can users create playlists? Playlists could be informative for the model.",
        "interviewer": "For simplicity, let's assume playlists don't exist.",
        "why_ask": "Shows you think about additional signal sources"
    },
    {
        "candidate": "How many videos are available on the platform?",
        "interviewer": "About 10 billion videos.",
        "why_ask": "10B means we CANNOT score every video per request -- need multi-stage pipeline"
    },
    {
        "candidate": "Can I assume recommendations should be served within 200 milliseconds?",
        "interviewer": "That sounds good.",
        "why_ask": "200ms budget forces architectural decisions (funnel approach)"
    }
]

print("=" * 70)
print("VIDEO RECOMMENDATION SYSTEM -- INTERVIEW CLARIFICATION")
print("=" * 70)
for i, qa in enumerate(interview_dialogue, 1):
    print(f"\n--- Question {i} ---")
    print(f"  Candidate:    {qa['candidate']}")
    print(f"  Interviewer:  {qa['interviewer']}")
    print(f"  [Why ask this: {qa['why_ask']}]")

print("\n" + "=" * 70)
print("KEY CONSTRAINTS:")
print("  - 10 BILLION videos (cannot score all per request)")
print("  - < 200ms latency (need fast retrieval)")
print("  - Homepage personalization (no immediate context signal)")
print("  - Global users, multiple languages")
print("=" * 70)

## 2. Framing the ML Problem

### Defining the ML Objective

The business says "increase engagement." But what does that MEAN for an ML model? Let's compare options.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# === ML Objective Comparison ===

objectives = {
    "Maximize Clicks": {
        "pros": "Easy to measure",
        "cons": "Leads to CLICKBAIT -- users click but don't watch",
        "analogy": "Picking food with the prettiest packaging -- looks good, tastes bad",
        "score": 3  # quality score out of 10
    },
    "Maximize Completed Videos": {
        "pros": "Good signal of satisfaction",
        "cons": "Biased toward SHORT videos (easier to finish a 10s clip)",
        "analogy": "Only recommending tiny snacks because people finish them quickly",
        "score": 5
    },
    "Maximize Watch Time": {
        "pros": "Captures deep engagement",
        "cons": "May favor addictive but low-quality content",
        "analogy": "Recommending infinite scroll content -- you watch but feel bad after",
        "score": 6
    },
    "Maximize Relevant Videos": {
        "pros": "Most control over quality signals",
        "cons": "Requires carefully defining 'relevance'",
        "analogy": "Picking food your friend will ACTUALLY enjoy eating",
        "score": 9
    }
}

print("=== ML Objective Comparison ===")
print(f"{'Objective':<30} {'Score':>6}  Description")
print("-" * 80)
for name, details in objectives.items():
    star = " <-- OUR CHOICE" if details['score'] == 9 else ""
    print(f"{name:<30} {details['score']:>4}/10  {details['pros']}{star}")
    print(f"{'':>38}  BUT: {details['cons']}")
    print(f"{'':>38}  Analogy: {details['analogy']}")
    print()

print("\n=== Our Definition of 'Relevant' ===")
print("A video is RELEVANT if the user:")
print("  1. Explicitly presses the LIKE button, OR")
print("  2. Watches at least HALF of the video")
print("\nThis combines explicit feedback (likes) with implicit feedback (watch time)")
print("and avoids the pitfalls of pure click or pure watch-time optimization.")

In [ ]:
# === Visualize the trade-offs ===

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

names = list(objectives.keys())
scores = [objectives[n]['score'] for n in names]
colors = ['#ff6b6b', '#ffa94d', '#69db7c', '#4dabf7']

bars = ax.barh(names, scores, color=colors, edgecolor='black', linewidth=0.5)

# Highlight the winner
bars[-1].set_edgecolor('#2b8a3e')
bars[-1].set_linewidth(3)

ax.set_xlabel('Quality Score (1-10)', fontsize=12)
ax.set_title('ML Objective Comparison for Video Recommendation', fontsize=14, fontweight='bold')
ax.set_xlim(0, 11)

# Add score labels
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{score}/10', va='center', fontsize=11, fontweight='bold')

# Add annotation for winner
ax.annotate('BEST CHOICE', xy=(9, 3), xytext=(6.5, 2.3),
            fontsize=12, fontweight='bold', color='#2b8a3e',
            arrowprops=dict(arrowstyle='->', color='#2b8a3e', lw=2))

plt.tight_layout()
plt.show()

### System Input and Output

```
INPUT:  A user (profile, history, context)
         |
         v
    [Video Recommendation System]
         |
         v
OUTPUT: Ranked list of videos sorted by predicted relevance scores
```

### ML Category: Hybrid Filtering

We combine collaborative filtering (who liked what) with content-based filtering (video features) in a sequential pipeline.

## 3. Types of Recommendation Systems

Before building our system, let's understand the three main approaches.

In [ ]:
# === Content-Based vs Collaborative Filtering ===

rec_types = {
    "Content-Based Filtering": {
        "how_it_works": [
            "1. User A engaged with ski videos X and Y",
            "2. Video Z is SIMILAR to X and Y (same tags, topic, features)",
            "3. Recommend video Z to User A"
        ],
        "analogy": "'You liked chocolate cake? Here are more chocolate desserts!'",
        "pros": ["Can recommend brand-new videos (no interaction data needed)",
                 "Captures unique interests of individual users"],
        "cons": ["Difficult to discover NEW interests (stuck in a bubble)",
                 "Requires manual feature engineering and domain knowledge"]
    },
    "Collaborative Filtering (CF)": {
        "how_it_works": [
            "1. Find User B who is similar to User A (watched same videos)",
            "2. Find video Z that User B watched but User A hasn't seen",
            "3. Recommend video Z to User A"
        ],
        "analogy": "'Your twin sister liked this video, so you probably will too!'",
        "pros": ["No domain knowledge needed (doesn't look at video content)",
                 "Can discover new interest areas",
                 "Efficient to compute"],
        "cons": ["COLD-START problem: can't recommend for new users or new videos",
                 "Struggles with niche/unique interests"]
    },
    "Hybrid Filtering (OUR CHOICE)": {
        "how_it_works": [
            "1. Use CF to quickly find a big pool of candidate videos",
            "2. Use content-based filtering to carefully RANK them",
            "3. Best of both worlds!"
        ],
        "analogy": "'Ask your twin for suggestions, then pick the ones that match YOUR exact taste'",
        "pros": ["Combines both data sources: interactions + video features",
                 "YouTube uses this exact approach (Google 2016 paper)"],
        "cons": ["More complex to build and maintain"]
    }
}

for name, details in rec_types.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"{'=' * 60}")
    print(f"  Analogy: {details['analogy']}")
    print(f"\n  How it works:")
    for step in details['how_it_works']:
        print(f"    {step}")
    print(f"\n  Pros:")
    for p in details['pros']:
        print(f"    + {p}")
    print(f"\n  Cons:")
    for c in details['cons']:
        print(f"    - {c}")

In [ ]:
# === Comparison Table Visualization ===

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

table_data = [
    ['Aspect', 'Content-Based', 'Collaborative Filtering'],
    ['Handle new videos', 'YES', 'NO'],
    ['Discover new interests', 'NO', 'YES'],
    ['No domain knowledge needed', 'NO', 'YES'],
    ['Efficiency', 'Lower', 'Higher'],
    ['Cold-start handling', 'Better', 'Worse'],
]

colors_table = []
for i, row in enumerate(table_data):
    if i == 0:
        colors_table.append(['#2b8a3e', '#2b8a3e', '#2b8a3e'])
    else:
        row_colors = ['#f8f9fa']
        for val in row[1:]:
            if val in ['YES', 'Better', 'Higher']:
                row_colors.append('#d3f9d8')
            elif val in ['NO', 'Worse', 'Lower']:
                row_colors.append('#ffe3e3')
            else:
                row_colors.append('#f8f9fa')
        colors_table.append(row_colors)

table = ax.table(cellText=table_data, cellColours=colors_table,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Header styling
for j in range(3):
    table[0, j].get_text().set_color('white')
    table[0, j].get_text().set_fontweight('bold')

ax.set_title('Content-Based vs Collaborative Filtering\n(Green = Advantage, Red = Disadvantage)',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nKey insight: The two methods are COMPLEMENTARY.")
print("That's why we use HYBRID filtering -- CF for candidate generation,")
print("content-based for ranking. Best of both worlds!")

## 4. Data Pipeline

### Available Data Sources

We have three types of data: **video data**, **user data**, and **user-video interaction data**. The interaction data is the gold mine -- it tells us not just WHAT users watched, but HOW they interacted.

In [ ]:
import pandas as pd

# === Video Data ===
video_data = pd.DataFrame({
    'video_id': [1, 2, 3, 4, 5],
    'length_seconds': [28, 300, 3600, 120, 900],
    'manual_tags': ['Dog, Family', 'Car, Oil', 'Ouli, Vlog', 'Python, Tutorial', 'Cooking, Italian'],
    'title': ['Our lovely dog playing!', 'How to change your car oil?',
              'Honeymoon to Bali', 'Python for Beginners', 'Making Perfect Pasta'],
    'likes': [138, 5, 2200, 890, 450],
    'views': [5300, 250, 255000, 42000, 18000],
    'language': ['en', 'en', 'en', 'en', 'it']
})

# === User Data ===
user_data = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'username': ['alice', 'bob', 'charlie', 'diana', 'eve'],
    'age': [25, 32, 19, 45, 28],
    'gender': ['F', 'M', 'M', 'F', 'F'],
    'city': ['San Francisco', 'New York', 'London', 'Tokyo', 'San Francisco'],
    'country': ['US', 'US', 'UK', 'JP', 'US']
})

# === User-Video Interaction Data (THE GOLD MINE) ===
interaction_data = pd.DataFrame({
    'user_id': [4, 2, 2, 6, 9, 8, 1, 1, 3, 5],
    'video_id': [18, 18, 6, 9, None, 6, 1, 3, 4, 2],
    'interaction_type': ['Like', 'Impression', 'Watch', 'Click',
                         'Search', 'Comment', 'Like', 'Watch', 'Watch', 'Impression'],
    'interaction_value': [None, '8 seconds', '46 minutes', None,
                         'Basics of clustering', 'Amazing video. Thanks',
                         None, '30 minutes', '2 minutes', '3 seconds']
})

print("=== Video Data ===")
print(video_data.to_string(index=False))
print("\n=== User Data ===")
print(user_data.to_string(index=False))
print("\n=== User-Video Interaction Data (THE GOLD MINE) ===")
print(interaction_data.to_string(index=False))

print("\n--- Why is interaction data the gold mine? ---")
print("It tells us not just WHAT users watched, but HOW they interacted:")
print("  - Likes = explicit positive signal (accurate but sparse)")
print("  - Watch time = implicit signal (abundant but noisy)")
print("  - Impressions = what they SAW but didn't engage with")
print("  - Searches = what they're LOOKING for")
print("  - Comments = deep engagement signal")

## 5. Feature Engineering

This is where the magic happens. We need to turn raw data into numbers that an ML model can learn from.

In [ ]:
import torch
import torch.nn as nn

# === Video Feature Engineering ===

print("=" * 70)
print("VIDEO FEATURE ENGINEERING")
print("=" * 70)

video_features = {
    "Video ID": {
        "type": "Categorical",
        "preparation": "Embedding layer (learned during training)",
        "why": "Unique identifier for collaborative filtering",
        "output_dim": "e.g., 64-dimensional vector"
    },
    "Duration": {
        "type": "Numerical",
        "preparation": "Raw value (possibly normalized)",
        "why": "Some users prefer short vs. long videos",
        "output_dim": "1 dimension"
    },
    "Language": {
        "type": "Categorical",
        "preparation": "Embedding layer",
        "why": "Users prefer specific languages",
        "output_dim": "e.g., 16-dimensional vector"
    },
    "Tags": {
        "type": "Text",
        "preparation": "Pre-trained CBOW embeddings (lightweight)",
        "why": "Describe video content topically",
        "output_dim": "e.g., 100-dimensional vector"
    },
    "Title": {
        "type": "Text",
        "preparation": "Pre-trained BERT embeddings (context-aware)",
        "why": "Rich semantic representation of content",
        "output_dim": "e.g., 768-dimensional vector"
    }
}

for feature, details in video_features.items():
    print(f"\n  {feature} ({details['type']})")
    print(f"    How to prepare: {details['preparation']}")
    print(f"    Why it matters: {details['why']}")
    print(f"    Output: {details['output_dim']}")

In [ ]:
# === User Feature Engineering ===

print("=" * 70)
print("USER FEATURE ENGINEERING")
print("=" * 70)

print("\n--- 1. Demographics ---")
print("  Age, Gender, Language -> Embedding layers")
print("  Why: Different demographics have different viewing preferences")

print("\n--- 2. Contextual Information ---")
context_features = {
    "Time of day": "A software engineer watches educational videos in the evening",
    "Device": "Mobile users may prefer shorter videos",
    "Day of week": "Weekend vs. weekday viewing habits differ greatly"
}
for feat, reason in context_features.items():
    print(f"  {feat}: {reason}")

print("\n--- 3. Historical Interactions (The Most Important!) ---")
history_features = {
    "Search history": "BERT-embed each query, then AVERAGE all query embeddings",
    "Liked videos": "Map video IDs to embeddings, then AVERAGE",
    "Watched videos": "Same approach as liked videos",
    "Impressions": "Videos shown but not clicked -- negative signal"
}
for feat, prep in history_features.items():
    print(f"  {feat}: {prep}")

print("\n=== KEY ENGINEERING INSIGHT ===")
print("All variable-length histories (searches, liked videos, watched videos)")
print("are converted to FIXED-SIZE vectors by AVERAGING their embeddings.")
print("\nThis is simple but effective. Why averaging?")
print("  - Each user has a DIFFERENT number of past interactions")
print("  - ML models need FIXED-SIZE inputs")
print("  - Averaging produces a single vector regardless of list length")

In [ ]:
# === Demo: Embedding Variable-Length History into Fixed-Size Vector ===

# Simulate: User has watched 5 videos, each with a 4-dim embedding
np.random.seed(42)
embedding_dim = 4

# Pretend these are the learned embeddings for 5 videos
watched_video_embeddings = np.random.randn(5, embedding_dim)
video_titles = ['Dog Playing', 'Cat Funny', 'Pet Care Tips', 'Dog Training', 'Animal Rescue']

print("=== Converting Variable-Length Watch History to Fixed-Size Vector ===")
print(f"\nUser watched {len(video_titles)} videos (variable length!)")
print(f"Each video has a {embedding_dim}-dim embedding:\n")

for title, emb in zip(video_titles, watched_video_embeddings):
    print(f"  '{title}': [{', '.join(f'{x:.2f}' for x in emb)}]")

# Average all embeddings
user_watch_embedding = watched_video_embeddings.mean(axis=0)
print(f"\nAVERAGE (fixed-size user watch embedding):")
print(f"  [{', '.join(f'{x:.2f}' for x in user_watch_embedding)}]")
print(f"\nThis single {embedding_dim}-dim vector represents the user's entire watch history!")
print("Whether they watched 5 videos or 5000, the output is always the same size.")

# Show that another user with different history length produces same-size output
watched_3 = np.random.randn(3, embedding_dim)
user2_embedding = watched_3.mean(axis=0)
print(f"\nAnother user watched only 3 videos:")
print(f"  Average embedding: [{', '.join(f'{x:.2f}' for x in user2_embedding)}]")
print(f"  Same {embedding_dim}-dim output! Models can now process both users identically.")

In [ ]:
# === Feature Engineering Architecture Diagram ===

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Feature Engineering Pipeline', fontsize=16, fontweight='bold', pad=20)

# --- Video Features (left side) ---
ax.add_patch(mpatches.FancyBboxPatch((0.5, 7), 5.5, 2.5, boxstyle='round,pad=0.15',
             facecolor='#e3f2fd', edgecolor='#1976d2', linewidth=2))
ax.text(3.25, 9.1, 'VIDEO FEATURES', ha='center', fontsize=12, fontweight='bold', color='#1976d2')
ax.text(1, 8.5, 'Video ID -> Embedding Layer (learned)', fontsize=9)
ax.text(1, 8.0, 'Duration -> Raw numerical value', fontsize=9)
ax.text(1, 7.5, 'Language -> Embedding Layer', fontsize=9)
ax.text(1, 7.0, 'Tags -> CBOW Embeddings', fontsize=9)

# --- User Demographics (right side, top) ---
ax.add_patch(mpatches.FancyBboxPatch((7.5, 7.5), 5.5, 2, boxstyle='round,pad=0.15',
             facecolor='#fce4ec', edgecolor='#c62828', linewidth=2))
ax.text(10.25, 9.1, 'USER DEMOGRAPHICS', ha='center', fontsize=12, fontweight='bold', color='#c62828')
ax.text(8, 8.5, 'Age, Gender -> Embedding Layers', fontsize=9)
ax.text(8, 8.0, 'Language -> Embedding Layer', fontsize=9)

# --- Contextual Features (right side, middle) ---
ax.add_patch(mpatches.FancyBboxPatch((7.5, 5), 5.5, 2, boxstyle='round,pad=0.15',
             facecolor='#fff3e0', edgecolor='#e65100', linewidth=2))
ax.text(10.25, 6.6, 'CONTEXTUAL FEATURES', ha='center', fontsize=12, fontweight='bold', color='#e65100')
ax.text(8, 6.0, 'Time of day, Device, Day of week', fontsize=9)
ax.text(8, 5.5, '-> Categorical embeddings', fontsize=9)

# --- Historical Interactions (bottom) ---
ax.add_patch(mpatches.FancyBboxPatch((0.5, 1.5), 12.5, 3, boxstyle='round,pad=0.15',
             facecolor='#e8f5e9', edgecolor='#2e7d32', linewidth=2))
ax.text(6.75, 4.1, 'HISTORICAL INTERACTIONS (Variable-Length -> Fixed-Size)',
        ha='center', fontsize=12, fontweight='bold', color='#2e7d32')
ax.text(1, 3.4, 'Search History: BERT-embed each query -> Average all embeddings', fontsize=9)
ax.text(1, 2.8, 'Liked Videos: Video ID -> Embedding -> Average all embeddings', fontsize=9)
ax.text(1, 2.2, 'Watched Videos: Video ID -> Embedding -> Average all embeddings', fontsize=9)
ax.text(1, 1.6, 'Impressions: Video ID -> Embedding -> Average all embeddings', fontsize=9)

# Arrows pointing down to model
ax.annotate('', xy=(6.75, 1.0), xytext=(6.75, 1.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(6.75, 0.5, 'CONCATENATE ALL -> Feed to Model', ha='center',
        fontsize=12, fontweight='bold', color='black',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black', linewidth=2))

plt.tight_layout()
plt.show()

## 6. Multi-Stage Serving Pipeline

### The Funnel: From 10 Billion to 50

This is the MOST IMPORTANT architectural decision. You cannot score every single video for every user in 200ms. Instead, use a funnel that progressively narrows down candidates.

In [ ]:
# === The Recommendation Funnel ===

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis('off')

# Title
ax.text(6, 9.5, 'The Video Recommendation Funnel', ha='center',
        fontsize=16, fontweight='bold')

# Funnel layers (wider at top, narrower at bottom)
funnel_stages = [
    {'y': 8.0, 'width': 10, 'label': '10 BILLION Videos',
     'color': '#e3f2fd', 'detail': 'All videos on the platform'},
    {'y': 6.5, 'width': 7, 'label': 'Candidate Generation',
     'color': '#bbdefb', 'detail': 'Fast & rough: Two-tower + ANN -> ~1,000-10,000'},
    {'y': 5.0, 'width': 5, 'label': 'Scoring / Ranking',
     'color': '#90caf9', 'detail': 'Slow & precise: Rich features -> ~100-500'},
    {'y': 3.5, 'width': 3.5, 'label': 'Re-Ranking',
     'color': '#64b5f6', 'detail': 'Business rules, diversity, freshness -> ~20-50'},
    {'y': 2.0, 'width': 2.5, 'label': 'FINAL: 20-50 Videos',
     'color': '#42a5f5', 'detail': 'Displayed on user homepage'}
]

for stage in funnel_stages:
    left = 6 - stage['width'] / 2
    rect = mpatches.FancyBboxPatch(
        (left, stage['y'] - 0.4), stage['width'], 0.8,
        boxstyle='round,pad=0.1',
        facecolor=stage['color'], edgecolor='#1565c0', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(6, stage['y'] + 0.05, stage['label'], ha='center', fontsize=11, fontweight='bold')
    ax.text(6, stage['y'] - 0.25, stage['detail'], ha='center', fontsize=9, style='italic')

# Arrows between stages
for i in range(len(funnel_stages) - 1):
    y_start = funnel_stages[i]['y'] - 0.4
    y_end = funnel_stages[i+1]['y'] + 0.4
    ax.annotate('', xy=(6, y_end), xytext=(6, y_start),
                arrowprops=dict(arrowstyle='->', lw=2, color='#1565c0'))

# Speed vs. accuracy labels
ax.text(0.5, 6.5, 'FAST\nLow accuracy\n(OK with false\npositives)', ha='center',
        fontsize=9, color='#2e7d32', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#e8f5e9', edgecolor='#2e7d32'))
ax.text(0.5, 3.5, 'SLOW\nHigh accuracy\n(Rich features,\nprecise scores)', ha='center',
        fontsize=9, color='#c62828', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#fce4ec', edgecolor='#c62828'))

ax.annotate('', xy=(0.5, 4.2), xytext=(0.5, 6.0),
            arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# Latency budget
ax.text(11.5, 5, 'Total Budget:\n< 200ms', ha='center',
        fontsize=11, fontweight='bold', color='#d32f2f',
        bbox=dict(boxstyle='round', facecolor='#ffebee', edgecolor='#d32f2f', linewidth=2))

plt.tight_layout()
plt.show()

In [ ]:
# === Stage-by-Stage Explanation ===

stages = {
    "Stage 1: Candidate Generation": {
        "goal": "Narrow BILLIONS to THOUSANDS",
        "priority": "Speed over accuracy (false positives are OK)",
        "model": "Two-tower neural network in CF mode",
        "how": [
            "1. Compute user embedding from user tower",
            "2. Use ANN (Approximate Nearest Neighbor) to find top-k similar video embeddings",
            "3. Return ranked candidates"
        ],
        "multiple_generators": [
            "CF-based: 'similar users liked these'",
            "Trending/popular: 'everyone is watching this'",
            "Location-based: 'popular in your area'",
            "New videos: 'fresh content for exploration'"
        ]
    },
    "Stage 2: Scoring (Ranking)": {
        "goal": "Precisely score each candidate with rich features",
        "priority": "Accuracy over speed (only thousands of candidates now)",
        "model": "Content-based ranking model (heavy model with many features)",
        "how": [
            "1. Take user features + video features as input",
            "2. Predict relevance score for each candidate",
            "3. Sort by score"
        ]
    },
    "Stage 3: Re-Ranking": {
        "goal": "Apply business rules and post-processing",
        "priority": "Business requirements and user experience",
        "rules": [
            "Remove inappropriate/flagged content",
            "Enforce diversity (don't show 10 cooking videos in a row!)",
            "Boost fresh/trending content",
            "Apply regional/legal restrictions",
            "Balance exploration vs. exploitation"
        ]
    }
}

for stage_name, details in stages.items():
    print(f"\n{'=' * 60}")
    print(f"  {stage_name}")
    print(f"{'=' * 60}")
    print(f"  Goal: {details['goal']}")
    print(f"  Priority: {details['priority']}")
    if 'model' in details:
        print(f"  Model: {details['model']}")
    if 'how' in details:
        print("  How:")
        for step in details['how']:
            print(f"    {step}")
    if 'multiple_generators' in details:
        print("  Multiple generators in parallel:")
        for gen in details['multiple_generators']:
            print(f"    - {gen}")
    if 'rules' in details:
        print("  Business rules:")
        for rule in details['rules']:
            print(f"    - {rule}")

## 7. Evaluation Metrics

In [ ]:
# === Offline Metrics ===

def precision_at_k(y_true, y_scores, k):
    """
    Precision@k: What fraction of the top-k recommended videos are relevant?
    
    12-year-old version:
    'Out of the top 5 videos I recommended, how many did you actually like?'
    """
    sorted_indices = np.argsort(-y_scores)[:k]
    relevant_in_top_k = np.sum(np.array(y_true)[sorted_indices])
    return relevant_in_top_k / k


def average_precision(y_true, y_scores):
    """
    Average Precision: Like precision@k but considers WHERE relevant items appear.
    Rewards putting relevant items near the TOP of the list.
    
    12-year-old version:
    'Not just HOW MANY good videos, but were the good ones at the TOP?'
    """
    sorted_indices = np.argsort(-y_scores)
    y_true_sorted = np.array(y_true)[sorted_indices]
    
    precisions = []
    relevant_count = 0
    for i, is_relevant in enumerate(y_true_sorted):
        if is_relevant == 1:
            relevant_count += 1
            precisions.append(relevant_count / (i + 1))
    return np.mean(precisions) if precisions else 0.0


def compute_diversity(embeddings):
    """
    Diversity: How different are the recommended videos from each other?
    Low average cosine similarity = high diversity.
    
    12-year-old version:
    'Did we recommend 10 dog videos, or a mix of dogs, cooking, and sports?'
    """
    from sklearn.metrics.pairwise import cosine_similarity
    sim_matrix = cosine_similarity(embeddings)
    n = len(embeddings)
    # Average pairwise similarity (exclude diagonal)
    total_sim = (np.sum(sim_matrix) - n) / (n * (n - 1))
    return 1 - total_sim  # Higher = more diverse


# Demo with a simple example
y_true = [1, 0, 1, 0, 0, 1, 0, 0, 0, 1]  # 4 relevant out of 10
y_scores = [0.95, 0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25, 0.15, 0.05]  # Our predictions

print("=== Offline Metrics Demo ===")
print(f"\nGround truth:  {y_true}")
print(f"Model scores:  {y_scores}")

for k in [1, 3, 5, 10]:
    p = precision_at_k(y_true, y_scores, k)
    print(f"\n  Precision@{k}: {p:.2f}")
    print(f"    (Out of top {k} recommended, {int(p*k)} were relevant)")

ap = average_precision(y_true, y_scores)
print(f"\n  Average Precision: {ap:.4f}")

# Diversity demo
video_embeddings = np.random.randn(5, 32)  # 5 recommended videos, 32-dim
div = compute_diversity(video_embeddings)
print(f"\n  Diversity Score: {div:.4f} (higher = more diverse)")
print("  Important: Diverse but irrelevant is USELESS. Always pair with relevance!")

In [ ]:
# === Online Metrics ===

online_metrics = {
    "CTR (Click-Through Rate)": {
        "formula": "clicked_videos / recommended_videos",
        "analogy": "How many flyers people pick up from a stack",
        "caveat": "Cannot detect clickbait! High CTR != good recommendations"
    },
    "Completed Videos": {
        "formula": "Count of fully watched recommended videos",
        "analogy": "How many meals people finish completely",
        "caveat": "Biased toward short content (easier to finish a 10s clip)"
    },
    "Total Watch Time": {
        "formula": "Sum of time spent on recommended videos",
        "analogy": "Total hours of entertainment provided",
        "caveat": "Core engagement signal -- directly measures platform value"
    },
    "Explicit Feedback": {
        "formula": "Count of likes/dislikes on recommended videos",
        "analogy": "Star ratings -- most accurate but few people bother",
        "caveat": "Most accurate but very sparse (few users click like/dislike)"
    }
}

print("=== Online Metrics ===")
for name, details in online_metrics.items():
    print(f"\n  {name}")
    print(f"    Formula: {details['formula']}")
    print(f"    Analogy: {details['analogy']}")
    print(f"    Caveat:  {details['caveat']}")

# Simulate online metrics
np.random.seed(42)
n_users = 10000
n_recs_per_user = 20
total_recs = n_users * n_recs_per_user

clicks = int(total_recs * 0.08)  # 8% CTR
completed = int(clicks * 0.35)   # 35% of clicked videos completed
watch_hours = clicks * 4.5       # Avg 4.5 min per clicked video
likes = int(clicks * 0.05)       # 5% of clicked videos get liked

print(f"\n=== Simulated Online Results ===")
print(f"  {n_users:,} users x {n_recs_per_user} recs = {total_recs:,} total recommendations")
print(f"  CTR: {clicks/total_recs*100:.1f}%  ({clicks:,} clicks)")
print(f"  Completed: {completed:,} videos")
print(f"  Total watch: {watch_hours:,.0f} minutes")
print(f"  Explicit likes: {likes:,} ({likes/clicks*100:.1f}% of clicked)")

## 8. Cold-Start Problem Solutions

In [ ]:
# === Cold-Start Problem Visualization ===

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

scenarios = [
    {
        "title": "New User, No History",
        "problem": "No watch/like history",
        "solution": "Use demographics\n(age, country, language)\nin two-tower model",
        "color": "#e3f2fd"
    },
    {
        "title": "New Video, No Interactions",
        "problem": "No one has watched it yet",
        "solution": "Use content features\n(title, tags, duration)\nin content-based filtering",
        "color": "#fff3e0"
    },
    {
        "title": "Completely Cold\n(New User + New Platform)",
        "problem": "No data at all",
        "solution": "Show popular/trending\nvideos as fallback\n(non-personalized)",
        "color": "#fce4ec"
    }
]

for ax_i, scenario in zip(axes, scenarios):
    ax_i.set_xlim(0, 10)
    ax_i.set_ylim(0, 10)
    ax_i.axis('off')
    
    # Title
    ax_i.text(5, 9, scenario['title'], ha='center', fontsize=11, fontweight='bold')
    
    # Problem box
    ax_i.add_patch(mpatches.FancyBboxPatch((1, 6), 8, 2, boxstyle='round,pad=0.15',
                   facecolor='#ffebee', edgecolor='#c62828', linewidth=2))
    ax_i.text(5, 7.3, 'PROBLEM', ha='center', fontsize=10, fontweight='bold', color='#c62828')
    ax_i.text(5, 6.6, scenario['problem'], ha='center', fontsize=9)
    
    # Arrow
    ax_i.annotate('', xy=(5, 5), xytext=(5, 6),
                  arrowprops=dict(arrowstyle='->', lw=2, color='gray'))
    
    # Solution box
    ax_i.add_patch(mpatches.FancyBboxPatch((1, 1.5), 8, 3, boxstyle='round,pad=0.15',
                   facecolor=scenario['color'], edgecolor='#1565c0', linewidth=2))
    ax_i.text(5, 4, 'SOLUTION', ha='center', fontsize=10, fontweight='bold', color='#1565c0')
    ax_i.text(5, 2.8, scenario['solution'], ha='center', fontsize=9)

fig.suptitle('Cold-Start Problem Solutions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Interview Cheat Sheet

In [ ]:
# === The 7-Step Framework ===

framework = [
    ("Clarify Requirements", "1 min",
     "Business objective, type of recommendation, scale, latency"),
    ("Frame the ML Problem", "2 min",
     "ML objective (maximize relevant videos), I/O, hybrid filtering"),
    ("Data and Features", "3 min",
     "Video features, user features (demographics + context + history)"),
    ("Model Architecture", "5 min",
     "Matrix factorization vs. two-tower, WALS, cross-entropy loss"),
    ("Evaluation", "3 min",
     "Offline (precision@k, mAP, diversity) + Online (CTR, watch time)"),
    ("Serving Pipeline", "5 min",
     "Candidate generation -> scoring -> re-ranking (the funnel!)"),
    ("Deep Dives", "remaining",
     "Cold start, ANN search, scalability, A/B testing")
]

print("=" * 70)
print("     INTERVIEW CHEAT SHEET: VIDEO RECOMMENDATION SYSTEM")
print("=" * 70)
print("\n  The 7-Step Framework (45-minute interview)\n")

total_time = 0
for i, (step, time, content) in enumerate(framework, 1):
    t = int(time.split()[0]) if time[0].isdigit() else 26
    total_time += t
    bar = '#' * (t * 2)
    print(f"  Step {i}: {step} ({time})")
    print(f"    [{bar}]")
    print(f"    Key points: {content}\n")

print("\n" + "=" * 70)
print("  KEY PHRASES TO USE IN THE INTERVIEW")
print("=" * 70)
key_phrases = [
    '"We use HYBRID filtering -- CF for candidate generation, content-based for ranking"',
    '"The funnel narrows 10B videos to ~50 in under 200ms"',
    '"We define relevance as: liked OR watched at least half"',
    '"Variable-length histories are averaged into fixed-size embeddings"',
    '"Two-tower handles cold start via user features, unlike matrix factorization"',
    '"ANN search gives us sub-linear retrieval time for candidate generation"',
    '"We use multiple candidate generators in parallel for diversity"',
    '"WALS converges faster than SGD and is parallelizable"'
]
for phrase in key_phrases:
    print(f"  -> {phrase}")

print("\n" + "=" * 70)
print("  COMMON FOLLOW-UP QUESTIONS")
print("=" * 70)
followups = [
    ("Why not just use one model?",
     "Scoring 10B videos per request is infeasible in 200ms. Multi-stage pipeline trades speed/accuracy."),
    ("How do you handle cold-start?",
     "Two-tower uses user features (demographics). New videos use content features. Fallback: trending."),
    ("Why WALS over SGD?",
     "WALS converges faster and is parallelizable. Alternates fixing U/V as least squares problems."),
    ("How does ANN search work?",
     "FAISS/ScaNN build index structures (IVF, HNSW) for sub-linear nearest neighbor search."),
    ("How do you handle data imbalance?",
     "Negative sampling, class weighting, oversampling positives. Typically 3:1 to 10:1 neg:pos ratio."),
    ("Why combine explicit + implicit feedback?",
     "Explicit (likes) is accurate but sparse. Implicit (watch time) is abundant but noisy. Best of both.")
]
for q, a in followups:
    print(f"\n  Q: {q}")
    print(f"  A: {a}")

## Key Takeaways

1. **Multi-stage pipeline is essential** -- you cannot score 10B videos in 200ms with one model
2. **Hybrid filtering** combines the best of CF (candidate generation) and content-based (ranking)
3. **Define relevance carefully** -- "liked OR watched at least half" avoids clickbait and short-video bias
4. **Feature engineering is critical** -- video features, user demographics, context, and averaged historical embeddings
5. **Two-tower beats matrix factorization** for quality but is slower at serving
6. **Cold-start has solutions** -- demographics for new users, content features for new videos, trending as fallback
7. **Evaluation needs both offline and online metrics** -- precision@k/mAP/diversity + CTR/watch time/feedback

---

### Next Notebooks
- **Notebook 2**: Deep dive into candidate generation (matrix factorization, two-tower, ANN/FAISS)
- **Notebook 3**: Deep dive into ranking models (content-based ranking, diversity, re-ranking)
- **Notebook 4**: Complete mock interview walkthrough with timing and diagrams

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)